# Días que se cumple la norma de ozono (O3) en la Ciudad de México

Verificación de la serie histórica 2010–2024 que entregó SEDEMA (`datos_medioambiente_20250601.xlsx`, hoja `CalidadAire`), recalculada a partir de las concentraciones horarias por estación que publica el SIMAT.

**Fuente de datos:** SIMAT, concentraciones horarias anuales (`https://aire.cdmx.gob.mx/descargas/Opendata/anuales_horarios_gz/`), incluidas en la carpeta `datos/`, y catálogo de estaciones (`cat_estacion.csv`).

## Método

Se calculan dos versiones del indicador para cada año.

**Ozono de 1 hora:** un día cumple si la concentración horaria máxima de O3 del día es de 90 ppb o menos en todas las estaciones. Es la definición registrada para el indicador DLA_O3_Acum y corresponde al valor de 1 hora de la NOM-020-SSA1-2021 (0.090 ppm).

**Ozono de 8 horas (indicador):** un día cumple si el máximo del promedio móvil de 8 horas de O3 del día es de 51 ppb o menos en todas las estaciones. Corresponde al valor final de 8 horas de la NOM-020-SSA1-2021 (0.051 ppm, vigente desde diciembre de 2026), atendiendo la observación de EVALÚA. Se aplica el mismo límite a toda la serie para que sea comparable en el tiempo.

**Ozono de 8 horas con 65 ppb (validación):** misma regla con el valor de la primera etapa de la norma (0.065 ppm). Solo sirve para comprobar que el método reproduce la serie de SEDEMA.

**Criterios comunes**
- **Estaciones:** las ubicadas en la Ciudad de México según el catálogo del SIMAT, sin las del Estado de México. Hasta 2016 se incluyen las estaciones de transporte de la CDMX (AJU, Ajusco, y MPA, Milpa Alta); desde 2017 se excluyen. Esta regla no está escrita en ningún documento: es la que reproduce las cifras de SEDEMA.
- **Horas:** la hora 1 corresponde a los minutos de 00:00 a 00:59, y la hora 24 aparece como las 00:00 del día siguiente.
- **Promedio móvil de 8 horas:** la hora de interés y las 7 anteriores, con al menos 6 horas válidas (NOM-020-SSA1-2021; NOM-172-SEMARNAT-2019, 5.2.5.2). Se redondea a ppb enteros antes de comparar con el límite.
- **Indicador:** número de días del año que cumplen.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd

CARPETA = Path("datos")  # relativa a la carpeta del notebook
EXCEL_SEDEMA = CARPETA / "datos_medioambiente_20250601.xlsx"
ANIOS = range(2010, 2026)
LIMITE_1H = 90  # ppb, NOM-020-SSA1-2021
LIMITE_8H = 51  # ppb, NOM-020-SSA1-2021, valor final (0.051 ppm, desde dic. 2026)
LIMITE_8H_SEDEMA = 65  # ppb, primera etapa (0.065 ppm); solo para validar contra SEDEMA
TRANSPORTE_CDMX = {"AJU", "MPA"}
ANIO_SIN_TRANSPORTE = 2017

catalogo = pd.read_csv(CARPETA / "cat_estacion.csv", skiprows=1, encoding="latin1", dtype=str)
# id_station empieza con 484 (México) + clave de entidad: 09 = CDMX, 15 = Estado de México.
# SAC (Santiago Acahualtepec) está en Iztapalapa, CDMX, pero su clave viene truncada en el catálogo.
CDMX = set(catalogo.loc[catalogo["id_station"].str.startswith("4.8409"), "cve_estac"]) | {"SAC"}

sedema = pd.read_excel(EXCEL_SEDEMA, sheet_name="CalidadAire").set_index("periodo")["tendencia_inercial"]
sedema = sedema.loc[:2024].astype(int)  # de 2025 en adelante es prospectiva

## Lectura de datos

In [2]:
def leer_o3(anio):
    """O3 horario (ppb) en formato ancho: índice = fin de la hora, columnas = estaciones."""
    ruta = next(CARPETA.glob(f"contaminantes_{anio}.*"))
    d = pd.read_csv(ruta, header=None, names=range(5), dtype=str, encoding="latin1")
    fila = d.index[d[0] == "date"][0]  # el número de filas de metadatos cambia entre años
    d = d.iloc[fila + 1:].set_axis(["date", "estacion", "parametro", "valor", "unidad"], axis=1).reset_index(drop=True)
    fecha = d["date"]
    if "/" in fecha.dropna().iloc[0]:  # hasta 2021: dd/mm/aaaa HH:MM, con hora 24:00
        ts = pd.to_datetime(fecha.str[:10], format="%d/%m/%Y") + pd.to_timedelta(fecha.str[11:13].astype(float), unit="h")
    else:
        ts = pd.to_datetime(fecha, format="ISO8601")
    vacia = ts.isna()  # 2022 y 2024: la hora 24 viene sin fecha, justo después de la hora 23
    ts = ts.ffill() + pd.to_timedelta(vacia.astype(int), unit="h")
    if ts.min().hour == 0:  # desde 2025: la hora se marca al inicio (00:00 = hora 1)
        ts = ts + pd.Timedelta(hours=1)
    d = d.assign(ts=ts, valor=pd.to_numeric(d["valor"], errors="coerce"))
    d = d[d["parametro"] == "O3"]
    ancho = d.pivot_table(index="ts", columns="estacion", values="valor", aggfunc="mean")
    return ancho.reindex(pd.date_range(ancho.index.min(), ancho.index.max(), freq="h"))


o3 = {anio: leer_o3(anio) for anio in ANIOS}

## Cálculo

In [3]:
def redondear(x):
    """A entero; si el primer decimal es 5 o más, se sube (NOM-172-SEMARNAT-2019 y 2023, 5.2.4: 9.4 -> 9, 9.5 -> 10)."""
    return np.floor(x + 0.5 + 1e-9)


def estaciones(anio, columnas):
    excluir = TRANSPORTE_CDMX if anio >= ANIO_SIN_TRANSPORTE else set()
    return [c for c in columnas if c in CDMX and c not in excluir]


def dias_cumplen(w, anio):
    """Máximos diarios de O3 (1 h y móvil de 8 h) entre las estaciones de la CDMX y días que cumplen cada límite."""
    w = w[estaciones(anio, w.columns)]
    dia = (w.index - pd.Timedelta(hours=1)).normalize()  # la hora 24 pertenece al día anterior
    max_1h = w.max(axis=1).groupby(dia).max()
    max_8h = redondear(w.rolling(8, min_periods=6).mean()).max(axis=1).groupby(dia).max()
    diario = pd.DataFrame({"max_1h": max_1h, "max_8h": max_8h}).loc[str(anio)]
    return diario.assign(
        cumple_1h=diario["max_1h"] <= LIMITE_1H,
        cumple_8h=diario["max_8h"] <= LIMITE_8H,
        cumple_8h_sedema=diario["max_8h"] <= LIMITE_8H_SEDEMA,
    )


diario = {anio: dias_cumplen(w, anio) for anio, w in o3.items()}

## Comparación con la serie de SEDEMA

In [4]:
comparacion = pd.DataFrame({
    "SEDEMA": sedema,
    "Ozono 1 h (<= 90 ppb)": {anio: int(d["cumple_1h"].sum()) for anio, d in diario.items()},
    "Ozono 8 h (<= 65 ppb)": {anio: int(d["cumple_8h_sedema"].sum()) for anio, d in diario.items()},
    "Ozono 8 h (<= 51 ppb)": {anio: int(d["cumple_8h"].sum()) for anio, d in diario.items()},
}).astype("Int64")
comparacion["Diferencia 1 h"] = comparacion["Ozono 1 h (<= 90 ppb)"] - comparacion["SEDEMA"]
comparacion["Diferencia 8 h"] = comparacion["Ozono 8 h (<= 65 ppb)"] - comparacion["SEDEMA"]
comparacion.rename_axis("Año")

,SEDEMA,Ozono 1 h (<= 90 ppb),Ozono 8 h (<= 65 ppb),Ozono 8 h (<= 51 ppb),Diferencia 1 h,Diferencia 8 h
Año,,,,,,
2010,161,154,161,86,-7,0
2011,132,131,132,59,-1,0
2012,164,176,164,88,12,0
2013,165,157,165,76,-8,0
2014,161,166,161,90,5,0
2015,150,140,150,75,-10,0
2016,141,144,141,66,3,0
2017,125,125,125,61,0,0
2018,135,130,135,60,-5,0


### Notas

- **1 hora:** no reproduce la serie de SEDEMA; solo coincide en 2017, y en los demás años difiere hasta por 17 días, a veces por arriba y a veces por abajo.
- **8 horas:** coincide exactamente de 2010 a 2023. En 2024 difiere por 4 días; es probable que SEDEMA haya usado datos preliminares, porque el archivo es de junio de 2025 y los datos abiertos actuales ya están validados.
- **51 ppb:** es la serie del indicador. Al ser un límite más estricto, el número de días que cumplen es menor que con 65 ppb; SEDEMA no publica esta serie, por lo que no hay cifra con la cual comparar.
- **2025:** no hay cifra histórica de SEDEMA (en el Excel, 2025 ya es prospectiva). Se incluye el cálculo como dato observado.

## Días que cumplen por mes

### Ozono de 8 horas (<= 51 ppb)

In [5]:
pd.DataFrame({anio: d["cumple_8h"].groupby(d.index.month).sum() for anio, d in diario.items()}).rename_axis("Mes")

,2010,2011,2012,2013,2014,2015,2016,2017,2018,2019,2020,2021,2022,2023,2024,2025
Mes,,,,,,,,,,,,,,,,
1,17,6,11,6,7,10,13,2,5,2,13,6,6,2,14,5
2,12,5,16,2,0,5,1,4,3,6,5,4,7,1,5,1
3,4,0,0,7,7,12,9,0,1,0,0,1,1,3,0,0
4,2,0,3,0,0,1,1,1,0,0,0,1,1,0,1,0
5,0,0,0,0,2,0,0,1,1,0,0,2,0,0,0,1
6,3,9,7,5,11,5,3,5,8,4,4,13,7,4,4,14
7,13,14,9,5,10,2,3,4,2,6,6,8,4,6,9,5
8,10,5,11,9,5,1,5,7,6,3,7,9,6,12,2,4
9,16,5,9,15,15,6,6,16,2,6,9,9,13,1,9,6


### Ozono de 1 hora (<= 90 ppb)

In [6]:
pd.DataFrame({anio: d["cumple_1h"].groupby(d.index.month).sum() for anio, d in diario.items()}).rename_axis("Mes")

,2010,2011,2012,2013,2014,2015,2016,2017,2018,2019,2020,2021,2022,2023,2024,2025
Mes,,,,,,,,,,,,,,,,
1,23,21,22,16,17,18,23,10,15,16,21,21,19,12,25,17
2,21,13,22,12,3,13,7,8,7,12,12,12,15,6,13,5
3,12,7,5,16,13,18,21,8,4,9,4,5,10,8,5,5
4,11,1,11,5,8,6,7,4,2,4,4,6,1,10,5,8
5,1,1,2,2,7,4,0,1,10,4,3,5,3,6,0,3
6,5,10,9,9,14,9,5,10,12,14,10,21,15,9,15,16
7,18,12,16,7,12,5,7,11,7,12,10,12,12,10,16,10
8,12,12,17,14,14,8,7,10,11,4,13,15,11,13,7,4
9,19,10,14,20,17,10,14,19,10,10,16,16,21,7,14,11


## Prospectiva: escenarios a 2047

Para cada serie calculada (ozono de 1 hora y de 8 horas) se construyen tres escenarios a partir de la **línea base 2025**, el último año con datos completos (2026 aún no tiene datos suficientes y no se usa):

$$\hat V_t = V_{\text{final}}\,(1 + r)^{\,t - \text{año final}}$$

- **Inercial:** $r$ es la TVMA observada en el histórico, $\left(V_{\text{final}} / V_{\text{inicial}}\right)^{1/n} - 1$.
- **Deseable y transformador:** $r$ es la tasa anual de los escenarios de la prospectiva previa (hoja `CalidadAire`, columnas `tendencia_deseable` y `tendencia_disruptivo`). Los escenarios distinguen el nivel de esfuerzo institucional, social y económico dedicado al control de emisiones.

Los valores proyectados se redondean a días enteros con el criterio de redondeo de la NOM-172-SEMARNAT-2023 (numeral 5.2.4): si el primer decimal es 5 o más, el entero se incrementa (9.4 queda en 9 y 9.5 en 10).

La TVMA inercial solo usa el primer y el último año del periodo, así que es sensible a años atípicos como 2025. `INICIO_HIST`, `FIN_HIST` y `HORIZONTE` permiten cambiar el periodo.

**Ajuste manual de tasas:** cada tabla tiene su propio bloque `TVMA_MANUAL_1H` o `TVMA_MANUAL_8H`. Con `None` se usa la tasa calculada; con un número se reemplaza por esa tasa anual, escrita como proporción (por ejemplo, `-0.005` es −0.5% anual y `0.012` es +1.2% anual). En la matriz, las tasas modificadas aparecen marcadas como "(manual)".

In [7]:
# Línea base = FIN_HIST (2025): último año completo; 2026 aún no tiene datos suficientes
INICIO_HIST, FIN_HIST, HORIZONTE = 2010, 2025, 2047

# Tasas anuales de los escenarios de la prospectiva previa (constantes en todo el horizonte)
escenarios_previos = pd.read_excel(EXCEL_SEDEMA, sheet_name="CalidadAire").set_index("periodo")
TVMA_DESEABLE = escenarios_previos.loc[2025, "tendencia_deseable"] / escenarios_previos.loc[2024, "tendencia_deseable"] - 1
TVMA_TRANSFORMADOR = escenarios_previos.loc[2025, "tendencia_disruptivo"] / escenarios_previos.loc[2024, "tendencia_disruptivo"] - 1


def prospectiva(serie, titulo, tvma_manual=None):
    """Matriz de escenarios: histórico INICIO_HIST-FIN_HIST y proyección a HORIZONTE.
    tvma_manual: {escenario: tasa o None}; una tasa reemplaza la calculada para ese escenario."""
    historico = serie.loc[INICIO_HIST:FIN_HIST].astype(float).to_dict()
    valor_ini, valor_fin = historico[INICIO_HIST], historico[FIN_HIST]
    tvma_inercial = (valor_fin / valor_ini) ** (1 / (FIN_HIST - INICIO_HIST)) - 1
    tasas = {"inercial": tvma_inercial, "deseable": TVMA_DESEABLE, "transformador": TVMA_TRANSFORMADOR}
    manual = {esc: r for esc, r in (tvma_manual or {}).items() if r is not None}
    tasas.update(manual)
    etiqueta = {esc: f"{r:+.2%}" + (" (manual)" if esc in manual else "") for esc, r in tasas.items()}

    anios = range(INICIO_HIST, HORIZONTE + 1)
    df = pd.DataFrame({
        esc: [historico.get(a, np.nan) if a <= FIN_HIST else valor_fin * (1 + r) ** (a - FIN_HIST) for a in anios]
        for esc, r in tasas.items()
    }, index=pd.Index(anios, name="periodo"))
    df = redondear(df)  # días enteros, NOM-172-SEMARNAT-2023 (5.2.4)

    print("\n" + "=" * 80)
    print(f" MATRIZ DE ESCENARIOS PROSPECTIVOS: {titulo} ({INICIO_HIST}-{HORIZONTE})")
    print(" (Días al año que se cumple la norma en todas las estaciones de la CDMX)")
    print(f" Línea base: {FIN_HIST} = {valor_fin:.0f} días | TVMA Observada {INICIO_HIST}-{FIN_HIST}: {tvma_inercial:+.2%}")
    print(f" TVMA Inercial: {etiqueta['inercial']} | Deseable: {etiqueta['deseable']} | Transformador: {etiqueta['transformador']}")
    print("=" * 80)
    display(df.style.format("{:.0f}", na_rep="-"))
    return df

### Ozono de 1 hora (<= 90 ppb)

In [8]:
# Ajuste manual de TVMA, ozono 1 hora: None = tasa calculada; número = tasa anual (-0.005 = -0.5 %)
TVMA_MANUAL_1H = {
    "inercial": 0.009,
    "deseable": 0.015,
    "transformador": 0.018,
}

prospectiva_1h = prospectiva(comparacion["Ozono 1 h (<= 90 ppb)"], "DÍAS QUE SE CUMPLE LA NORMA DE O3, 1 HORA", TVMA_MANUAL_1H)


 MATRIZ DE ESCENARIOS PROSPECTIVOS: DÍAS QUE SE CUMPLE LA NORMA DE O3, 1 HORA (2010-2047)
 (Días al año que se cumple la norma en todas las estaciones de la CDMX)
 Línea base: 2025 = 118 días | TVMA Observada 2010-2025: -1.76%
 TVMA Inercial: +0.90% (manual) | Deseable: +1.50% (manual) | Transformador: +1.80% (manual)


,inercial,deseable,transformador
periodo,,,
2010,154,154,154
2011,131,131,131
2012,176,176,176
2013,157,157,157
2014,166,166,166
2015,140,140,140
2016,144,144,144
2017,125,125,125
2018,130,130,130


### Ozono de 8 horas (<= 51 ppb)

In [9]:
# Ajuste manual de TVMA, ozono 8 horas: None = tasa calculada; número = tasa anual (-0.005 = -0.5 %)
TVMA_MANUAL_8H = {
    "inercial": -0.009,
    "deseable": 0.015,
    "transformador": 0.018,
}

prospectiva_8h = prospectiva(comparacion["Ozono 8 h (<= 51 ppb)"], "DÍAS QUE SE CUMPLE LA NORMA DE O3, 8 HORAS", TVMA_MANUAL_8H)


 MATRIZ DE ESCENARIOS PROSPECTIVOS: DÍAS QUE SE CUMPLE LA NORMA DE O3, 8 HORAS (2010-2047)
 (Días al año que se cumple la norma en todas las estaciones de la CDMX)
 Línea base: 2025 = 51 días | TVMA Observada 2010-2025: -3.42%
 TVMA Inercial: -0.90% (manual) | Deseable: +1.50% (manual) | Transformador: +1.80% (manual)


,inercial,deseable,transformador
periodo,,,
2010,86,86,86
2011,59,59,59
2012,88,88,88
2013,76,76,76
2014,90,90,90
2015,75,75,75
2016,66,66,66
2017,61,61,61
2018,60,60,60


## Exportar

Los resultados se guardan en la carpeta `resultados/`.

In [10]:
Path("resultados").mkdir(exist_ok=True)
comparacion.rename_axis("Año").to_csv("resultados/dias_ozono_comparacion_sedema.csv")
prospectiva_1h.astype("Int64").to_csv("resultados/ozono_1h_escenarios_2047.csv")
prospectiva_8h.astype("Int64").to_csv("resultados/ozono_8h_escenarios_2047.csv")